# Test All CBAM Weights on sb_no_hr Test Set

This notebook evaluates all `best.pt` and `last.pt` under:
`/workspace/dl_project/SE_CBAM/yolov5_CBAM/runs/train`

Requested test set root:
`/workspace/dl_project/SE_CBAM/dataset/sb_no_hr/test`


## 1) Define Paths


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
from datetime import datetime

import yaml

PROJECT_ROOT = Path('/workspace/dl_project/SE_CBAM')
YOLO_ROOT = PROJECT_ROOT / 'yolov5_CBAM'
TRAIN_RUNS_DIR = YOLO_ROOT / 'runs' / 'train'
TEST_SPLIT_DIR = PROJECT_ROOT / 'dataset' / 'sb_no_hr' / 'test'

TRAIN_DATA_5CLS = YOLO_ROOT / 'data' / 'StudentWatch_no_hr.yaml'
TRAIN_DATA_6CLS = YOLO_ROOT / 'data' / 'StudentWatch.yaml'

EVAL_DATA_5CLS = PROJECT_ROOT / 'test_sb_no_hr_5cls.yaml'
EVAL_DATA_6CLS = PROJECT_ROOT / 'test_sb_no_hr_6cls.yaml'

EVAL_6CLS_ROOT = PROJECT_ROOT / '_eval_sb_no_hr_as_6cls'

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
PYTHON_EXEC = sys.executable

PROJECT_ROOT, YOLO_ROOT, TRAIN_RUNS_DIR, TEST_SPLIT_DIR, EVAL_DATA_5CLS, EVAL_DATA_6CLS, PYTHON_EXEC


(PosixPath('/workspace/dl_project/SE_CBAM'),
 PosixPath('/workspace/dl_project/SE_CBAM/yolov5_CBAM'),
 PosixPath('/workspace/dl_project/SE_CBAM/yolov5_CBAM/runs/train'),
 PosixPath('/workspace/dl_project/SE_CBAM/dataset/sb_no_hr/test'),
 PosixPath('/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml'),
 PosixPath('/workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml'),
 '/opt/conda/bin/python')

## 2) Check Paths


In [2]:
assert PROJECT_ROOT.exists(), f'PROJECT_ROOT not found: {PROJECT_ROOT}'
assert YOLO_ROOT.exists(), f'YOLO_ROOT not found: {YOLO_ROOT}'
assert TRAIN_RUNS_DIR.exists(), f'TRAIN_RUNS_DIR not found: {TRAIN_RUNS_DIR}'
assert TEST_SPLIT_DIR.exists(), f'TEST_SPLIT_DIR not found: {TEST_SPLIT_DIR}'
assert TRAIN_DATA_5CLS.exists(), f'TRAIN_DATA_5CLS not found: {TRAIN_DATA_5CLS}'
assert TRAIN_DATA_6CLS.exists(), f'TRAIN_DATA_6CLS not found: {TRAIN_DATA_6CLS}'

print('All key paths exist.')


All key paths exist.


## 3) Build Evaluation YAMLs for 5-class and 6-class Models


In [3]:
# Requested test images from sb_no_hr/test
TEST_IMAGES_DIR = TEST_SPLIT_DIR / 'images' if (TEST_SPLIT_DIR / 'images').exists() else TEST_SPLIT_DIR
TEST_LABELS_DIR = TEST_SPLIT_DIR / 'labels'

assert TEST_IMAGES_DIR.exists(), f'TEST_IMAGES_DIR not found: {TEST_IMAGES_DIR}'
assert TEST_LABELS_DIR.exists(), f'TEST_LABELS_DIR not found: {TEST_LABELS_DIR}'

# 5-class yaml: direct evaluation on requested test set.
with open(TRAIN_DATA_5CLS, 'r', encoding='utf-8') as f:
    data5 = yaml.safe_load(f)

data5['test'] = str(TEST_IMAGES_DIR.resolve())
if 'path' in data5:
    data5['path'] = str(Path(data5['path']).resolve())

with open(EVAL_DATA_5CLS, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data5, f, sort_keys=False, allow_unicode=True)

# 6-class yaml: same test images, but labels remapped from 5cls ids to 6cls ids.
# Mapping for sb_no_hr -> StudentWatch classes:
# 0->0 (bowing), 1->2 (learning), 2->3 (reading), 3->4 (using_phone), 4->5 (writing)
map_5_to_6 = {0: 0, 1: 2, 2: 3, 3: 4, 4: 5}

images_out = EVAL_6CLS_ROOT / 'test' / 'images'
labels_out = EVAL_6CLS_ROOT / 'test' / 'labels'
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

# Materialize image files for YOLO path->label pairing under this new root.
image_paths = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp'):
    image_paths.extend(sorted(TEST_IMAGES_DIR.glob(ext)))
assert image_paths, f'No images found under: {TEST_IMAGES_DIR}'

for src_img in image_paths:
    dst_img = images_out / src_img.name
    if not dst_img.exists():
        try:
            os.link(src_img, dst_img)
        except OSError:
            shutil.copy2(src_img, dst_img)

for src_lbl in sorted(TEST_LABELS_DIR.glob('*.txt')):
    dst_lbl = labels_out / src_lbl.name
    remapped_lines = []
    with open(src_lbl, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.split()
            cls_id = int(parts[0])
            if cls_id not in map_5_to_6:
                raise ValueError(f'Unexpected class id {cls_id} in {src_lbl}')
            parts[0] = str(map_5_to_6[cls_id])
            remapped_lines.append(' '.join(parts))
    with open(dst_lbl, 'w', encoding='utf-8') as f:
        if remapped_lines:
            f.write(chr(10).join(remapped_lines) + chr(10))

with open(TRAIN_DATA_6CLS, 'r', encoding='utf-8') as f:
    data6 = yaml.safe_load(f)

data6['path'] = str(EVAL_6CLS_ROOT.resolve())
# Make train/val/test all point to the same requested test images so check_dataset passes.
data6['train'] = 'test/images'
data6['val'] = 'test/images'
data6['test'] = 'test/images'

with open(EVAL_DATA_6CLS, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data6, f, sort_keys=False, allow_unicode=True)

print('5-class eval yaml:', EVAL_DATA_5CLS)
print('6-class eval yaml:', EVAL_DATA_6CLS)
print('6-class remapped root:', EVAL_6CLS_ROOT)
print('images in test set:', len(image_paths))



5-class eval yaml: /workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
6-class eval yaml: /workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml
6-class remapped root: /workspace/dl_project/SE_CBAM/_eval_sb_no_hr_as_6cls
images in test set: 33


## 4) Collect All Weights and Match Each Weight to Correct Eval YAML


In [4]:
weight_jobs = []
skipped_jobs = []

for exp_dir in sorted(TRAIN_RUNS_DIR.iterdir()):
    if not exp_dir.is_dir():
        continue

    weights_dir = exp_dir / 'weights'
    if not weights_dir.exists():
        continue

    opt_yaml = exp_dir / 'opt.yaml'
    if opt_yaml.exists():
        with open(opt_yaml, 'r', encoding='utf-8') as f:
            opt_cfg = yaml.safe_load(f)
        train_data_yaml = Path(opt_cfg.get('data', TRAIN_DATA_5CLS))
        if not train_data_yaml.is_absolute():
            train_data_yaml = (YOLO_ROOT / train_data_yaml).resolve()
    else:
        train_data_yaml = TRAIN_DATA_5CLS.resolve()

    with open(train_data_yaml, 'r', encoding='utf-8') as f:
        train_data_cfg = yaml.safe_load(f)

    nc_train = int(train_data_cfg['nc'])

    if nc_train == 5:
        eval_data_yaml = EVAL_DATA_5CLS
    elif nc_train == 6:
        eval_data_yaml = EVAL_DATA_6CLS
    else:
        skipped_jobs.append({'exp_name': exp_dir.name, 'nc_train': nc_train, 'train_data_yaml': str(train_data_yaml)})
        continue

    for weight_type in ('best', 'last'):
        weight_path = weights_dir / f'{weight_type}.pt'
        if weight_path.exists():
            weight_jobs.append(
                {
                    'exp_name': exp_dir.name,
                    'weight_type': weight_type,
                    'weight_path': weight_path.resolve(),
                    'train_data_yaml': train_data_yaml.resolve(),
                    'nc_train': nc_train,
                    'eval_data_yaml': Path(eval_data_yaml).resolve(),
                }
            )

assert weight_jobs, f'No best/last weights found under: {TRAIN_RUNS_DIR}'

print(f'Total weights to evaluate: {len(weight_jobs)}')
for i, job in enumerate(weight_jobs, 1):
    print(f"{i:02d}. {job['exp_name']:<30} {job['weight_type']:<4} nc={job['nc_train']} data={job['eval_data_yaml']}")

if skipped_jobs:
    print('Skipped jobs:', skipped_jobs)


Total weights to evaluate: 10
01. sb_cbam_baseline               best nc=6 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml
02. sb_cbam_baseline               last nc=6 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml
03. sb_cbam_no_hr                  best nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
04. sb_cbam_no_hr                  last nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
05. sb_cbam_no_hr_aug              best nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
06. sb_cbam_no_hr_aug              last nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
07. sb_cbam_no_hr_aug_weighted     best nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
08. sb_cbam_no_hr_aug_weighted     last nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
09. sb_cbam_no_hr_weighted         best nc=5 data=/workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
10. sb_cbam_no_hr_weighted        

## 5) Run `val.py` on Test Split for Every Weight


In [5]:
os.chdir(YOLO_ROOT)
print('Current working directory:', Path.cwd())

val_results = []

for job in weight_jobs:
    run_name = f"{job['exp_name']}_{job['weight_type']}_{RUN_TAG}"
    output_dir = YOLO_ROOT / 'runs' / 'test_eval_cbam' / run_name

    cmd = [
        str(PYTHON_EXEC), 'val.py',
        '--data', str(job['eval_data_yaml']),
        '--weights', str(job['weight_path']),
        '--img', '640',
        '--batch', '2',
        '--task', 'test',
        '--project', 'runs/test_eval_cbam',
        '--name', run_name,
        '--save-txt',
        '--save-conf',
        '--verbose',
    ]

    print('=' * 100)
    print(f"Evaluating: {job['exp_name']} ({job['weight_type']}) | nc={job['nc_train']}")
    print('Eval data yaml:', job['eval_data_yaml'])
    print('Command:', ' '.join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True, cwd=YOLO_ROOT)

    val_results.append(
        {
            'exp_name': job['exp_name'],
            'weight_type': job['weight_type'],
            'weight_path': str(job['weight_path']),
            'nc_train': job['nc_train'],
            'eval_data_yaml': str(job['eval_data_yaml']),
            'output_dir': str(output_dir),
            'returncode': result.returncode,
            'stdout': result.stdout,
            'stderr': result.stderr,
        }
    )

    print('Return code:', result.returncode)

print('Finished all evaluations.')


Current working directory: /workspace/dl_project/SE_CBAM/yolov5_CBAM
Evaluating: sb_cbam_baseline (best) | nc=6
Eval data yaml: /workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml
Command: /opt/conda/bin/python val.py --data /workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml --weights /workspace/dl_project/SE_CBAM/yolov5_CBAM/runs/train/sb_cbam_baseline/weights/best.pt --img 640 --batch 2 --task test --project runs/test_eval_cbam --name sb_cbam_baseline_best_20260417_123822 --save-txt --save-conf --verbose


Return code: 0
Evaluating: sb_cbam_baseline (last) | nc=6
Eval data yaml: /workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml
Command: /opt/conda/bin/python val.py --data /workspace/dl_project/SE_CBAM/test_sb_no_hr_6cls.yaml --weights /workspace/dl_project/SE_CBAM/yolov5_CBAM/runs/train/sb_cbam_baseline/weights/last.pt --img 640 --batch 2 --task test --project runs/test_eval_cbam --name sb_cbam_baseline_last_20260417_123822 --save-txt --save-conf --verbose
Return code: 0
Evaluating: sb_cbam_no_hr (best) | nc=5
Eval data yaml: /workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml
Command: /opt/conda/bin/python val.py --data /workspace/dl_project/SE_CBAM/test_sb_no_hr_5cls.yaml --weights /workspace/dl_project/SE_CBAM/yolov5_CBAM/runs/train/sb_cbam_no_hr/weights/best.pt --img 640 --batch 2 --task test --project runs/test_eval_cbam --name sb_cbam_no_hr_best_20260417_123822 --save-txt --save-conf --verbose
Return code: 0
Evaluating: sb_cbam_no_hr (last) | nc=5
Eval data yaml: /workspace/d

## 6) Parse Metrics and Print Summary


In [6]:
def extract_all_metrics(output_text: str):
    for line in output_text.splitlines():
        parts = line.strip().split()
        if len(parts) >= 7 and parts[0] == 'all':
            try:
                return {
                    'P': float(parts[3]),
                    'R': float(parts[4]),
                    'mAP50': float(parts[5]),
                    'mAP50_95': float(parts[6]),
                }
            except ValueError:
                continue
    return None

summary_rows = []
failed_rows = []

for item in val_results:
    combined = (item.get('stdout', '') or '') + chr(10) + (item.get('stderr', '') or '')
    metrics = extract_all_metrics(combined)

    if item['returncode'] == 0 and metrics is not None:
        summary_rows.append(
            {
                'exp_name': item['exp_name'],
                'weight_type': item['weight_type'],
                'nc_train': item['nc_train'],
                'P': metrics['P'],
                'R': metrics['R'],
                'mAP50': metrics['mAP50'],
                'mAP50_95': metrics['mAP50_95'],
                'output_dir': item['output_dir'],
            }
        )
    else:
        tail = chr(10).join(combined.strip().splitlines()[-20:])
        failed_rows.append(
            {
                'exp_name': item['exp_name'],
                'weight_type': item['weight_type'],
                'nc_train': item['nc_train'],
                'returncode': item['returncode'],
                'eval_data_yaml': item['eval_data_yaml'],
                'output_dir': item['output_dir'],
                'log_tail': tail,
            }
        )

summary_rows = sorted(summary_rows, key=lambda x: (x['exp_name'], x['weight_type']))

if summary_rows:
    header = f"{'exp_name':<30} {'w':<4} {'nc':<4} {'P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95':>10}"
    print(header)
    print('-' * len(header))
    for row in summary_rows:
        print(
            f"{row['exp_name']:<30} {row['weight_type']:<4} {row['nc_train']:<4} "
            f"{row['P']:>8.4f} {row['R']:>8.4f} {row['mAP50']:>8.4f} {row['mAP50_95']:>10.4f}"
        )
else:
    print('No successful runs parsed.')

print()
print(f"Successful: {len(summary_rows)} / {len(val_results)}")
print(f"Failed    : {len(failed_rows)} / {len(val_results)}")

if failed_rows:
    print()
    print('Failed jobs (with last log lines):')
    for row in failed_rows:
        print('=' * 100)
        print(f"{row['exp_name']} ({row['weight_type']}) | nc={row['nc_train']} | returncode={row['returncode']}")
        print(f"eval_data_yaml: {row['eval_data_yaml']}")
        print(f"output_dir: {row['output_dir']}")
        print(row['log_tail'])


exp_name                       w    nc          P        R    mAP50   mAP50-95
------------------------------------------------------------------------------
sb_cbam_baseline               best 6      0.9200   0.8820   0.9110     0.6960
sb_cbam_baseline               last 6      0.9070   0.8910   0.9080     0.6990
sb_cbam_no_hr                  best 5      0.9170   0.8970   0.9090     0.6990
sb_cbam_no_hr                  last 5      0.9130   0.8960   0.9090     0.7060
sb_cbam_no_hr_aug              best 5      0.9150   0.8990   0.9190     0.7020
sb_cbam_no_hr_aug              last 5      0.9140   0.8980   0.9100     0.6950
sb_cbam_no_hr_aug_weighted     best 5      0.9210   0.8870   0.9140     0.6940
sb_cbam_no_hr_aug_weighted     last 5      0.9240   0.8800   0.9180     0.7000
sb_cbam_no_hr_weighted         best 5      0.9190   0.8780   0.9170     0.7070
sb_cbam_no_hr_weighted         last 5      0.9210   0.8870   0.9130     0.7040

Successful: 10 / 10
Failed    : 0 / 10
